# 🔥 TCN Training - Google Colab

Notebook per addestrare la TCN con ottimizzazione Optuna.

**Supporta ripresa:** Sia Optuna che Final Training possono essere ripresi se interrotti.

## 1. 📦 Clone Repository

In [ ]:
import os
if not os.path.exists('/content/Progetto_deep_learning'):
    !git clone --branch training_branch --single-branch https://github.com/scorzaluca/Progetto_deep_learning.git
else:
    print("Repository già presente, skip clone")
    
!ls Progetto_deep_learning/

In [ ]:
%pip install optuna -q

import sys
sys.path.insert(0, '/content/Progetto_deep_learning')
print("Repository pronto!")

## 2. ⚙️ Configurazione

In [ ]:
import torch

# ===== CONFIGURAZIONE =====
DATA_PATH = "/content/Progetto_deep_learning/data/processed/preprocessed_ds.csv"
MODEL_NAME = "tcn"

# --- Optuna ---
N_TRIALS = 50
N_FOLDS = 1
OPTUNA_EPOCHS = 30
OPTUNA_PATIENCE = 7

# --- Studio Optuna (per ripresa) ---
STUDY_NAME = f"{MODEL_NAME}_colab"
STORAGE_PATH = f"/content/{MODEL_NAME}_optuna.db"
NEW_STUDY = True         # False = riprendi studio esistente

# --- Final Training ---
FINAL_EPOCHS = 100
FINAL_PATIENCE = 15
CHECKPOINT_PATH = f"/content/{MODEL_NAME}_checkpoint.pth"  # Per ripresa training
RESUME_TRAINING = False  # True = riprendi da checkpoint

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
print(f"\n--- Optuna ---")
print(f"Studio: {STUDY_NAME}")
print(f"Nuovo studio: {NEW_STUDY}")
print(f"\n--- Final Training ---")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Riprendi training: {RESUME_TRAINING}")

## 3. 📊 Carica Dati

In [ ]:
from src.config import SEED
from src.Utils import set_seed, load_data_and_folds

set_seed(SEED)

df, folds = load_data_and_folds(data_path=DATA_PATH)
print(f"\nDataset: {df.shape}")
print(f"Folds: {len(folds)}")

## 4. 🔍 Ottimizzazione Optuna

In [ ]:
from src.Tuning import OptunaOptimizer

config = {
    "n_trials": N_TRIALS,
    "n_folds": N_FOLDS,
    "epochs": OPTUNA_EPOCHS,
    "patience": OPTUNA_PATIENCE,
}

optimizer = OptunaOptimizer(
    model_name=MODEL_NAME,
    folds=folds,
    device=DEVICE,
    config=config,
    storage_path=STORAGE_PATH,
    verbose=False,
)

result = optimizer.optimize(
    study_name=STUDY_NAME,
    n_trials=N_TRIALS,
    new_study=NEW_STUDY,
)

print(f"\nBest MASE: {result['best_mase']:.4f}")
print(f"Best params: {result['best_params']}")

## 5. 🏋️ Final Training (con checkpoint)

In [ ]:
import os
import copy
import math
import torch.nn as nn
from torch.amp import autocast, GradScaler
from src.Training.engine import create_model, train_one_epoch, validate_one_epoch
from src.DataLoading import create_final_train_val_loaders
from src.config import TARGET_COL, NAIVE_MAE_FINAL_FOLD

best_params = result["best_params"]

# Final loaders
train_loader, val_loader, scaler = create_final_train_val_loaders(df, TARGET_COL)

# Create model
model = create_model(MODEL_NAME, best_params)
model.to(DEVICE)

# Optimizer e loss
lr = best_params.get("lr", 0.001)
weight_decay = best_params.get("weight_decay", 0.01)
grad_clip_norm = best_params.get("grad_clip_norm", 1.0)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.MSELoss()
use_amp = DEVICE.type == "cuda"
amp_scaler = GradScaler(enabled=use_amp) if use_amp else None

# History e stato
history = {"train_loss": [], "val_loss": [], "val_mase": [], "val_rmse": []}
best_mase = float("inf")
epochs_no_improve = 0
best_model_wts = copy.deepcopy(model.state_dict())
best_epoch = 0
start_epoch = 0

# --- RIPRESA DA CHECKPOINT ---
if RESUME_TRAINING and os.path.exists(CHECKPOINT_PATH):
    print(f"Caricamento checkpoint: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_mase = checkpoint["best_mase"]
    best_model_wts = checkpoint["best_model_wts"]
    history = checkpoint["history"]
    epochs_no_improve = checkpoint["epochs_no_improve"]
    print(f"Ripreso dall'epoca {start_epoch}, best MASE: {best_mase:.4f}")
else:
    print(f"Nuovo training da epoca 0")

print(f"\nTraining {MODEL_NAME.upper()} su {DEVICE}...")

# --- TRAINING LOOP ---
for epoch in range(start_epoch, FINAL_EPOCHS):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, loss_fn, DEVICE, grad_clip_norm, amp_scaler
    )
    val_loss, val_mae, val_rmse = validate_one_epoch(model, val_loader, loss_fn, DEVICE)
    current_mase = val_mae / NAIVE_MAE_FINAL_FOLD
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mase"].append(current_mase)
    history["val_rmse"].append(val_rmse)
    
    if (epoch + 1) % 5 == 0 or epoch == start_epoch:
        print(f"Epoch {epoch+1}/{FINAL_EPOCHS} | Train MSE: {train_loss:.6f} | Val MASE: {current_mase:.4f}")
    
    # Early stopping check
    if current_mase < best_mase:
        best_mase = current_mase
        epochs_no_improve = 0
        best_model_wts = copy.deepcopy(model.state_dict())
        best_epoch = epoch
    else:
        epochs_no_improve += 1
    
    # --- SALVA CHECKPOINT OGNI EPOCA ---
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_mase": best_mase,
        "best_model_wts": best_model_wts,
        "history": history,
        "epochs_no_improve": epochs_no_improve,
    }, CHECKPOINT_PATH)
    
    # Early stopping
    if epochs_no_improve >= FINAL_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}. Best MASE: {best_mase:.4f}")
        break

# Carica best weights
model.load_state_dict(best_model_wts)
print(f"\nTraining completato! Best MASE: {best_mase:.4f} (epoch {best_epoch+1})")

## 6. 💾 Salva Risultati

In [ ]:
import json

torch.save(model.state_dict(), f"{MODEL_NAME}_best_model.pth")

with open(f"{MODEL_NAME}_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

with open(f"{MODEL_NAME}_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("Salvati: model, params, history")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("MSE Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["val_mase"], color="green")
axes[1].axhline(y=1.0, color="red", linestyle="--")
axes[1].set_title("MASE")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{MODEL_NAME}_training.png", dpi=150)
plt.show()

In [ ]:
from google.colab import files

files.download(f"{MODEL_NAME}_best_model.pth")
files.download(f"{MODEL_NAME}_best_params.json")
files.download(f"{MODEL_NAME}_history.json")
files.download(f"{MODEL_NAME}_training.png")
files.download(STORAGE_PATH)
files.download(CHECKPOINT_PATH)